# 04B. Satellite Rainfall & Radar (Fixed Download)

In order to teach a model *when* and *where* a flood occurs, we need Historical Rainfall (the trigger) and Sentinel-1 SAR (the actual flood footprint observed from space).
This notebook downloads historical extreme events (e.g. Cyclone Idai 2019) to act as your training data for `Rainfall_mm` and `Flood_Target`.


In [1]:
import ee
import geopandas as gpd
import requests
import json
import os
from pathlib import Path

try:
    ee.Initialize()
except Exception as e:
    print('Please run notebook 02 to set up authentication first.')
    raise e

# Load Chikwawa Boundary using absolute paths
notebook_dir  = Path(os.path.abspath(''))
project_root  = notebook_dir.parent
boundary_path = project_root / 'data' / 'raw' / 'chikwawa_boundary.geojson'

chikwawa_gdf = gpd.read_file(boundary_path)
geojson = json.loads(chikwawa_gdf.to_json())
aoi = ee.FeatureCollection(geojson).geometry()


In [2]:
# 1. CHIRPS Daily Precipitation Data (Rainfall_mm)
start_date = '2019-03-01'
end_date = '2019-03-31'

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
            .filterDate(start_date, end_date) \
            .select('precipitation')

total_rainfall = chirps.sum().clip(aoi)


In [3]:
# 2. Flood Mapping using Sentinel-1 Radar
collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .select('VV')

before_flood = collection.filterDate('2019-01-01', '2019-02-28').min().clip(aoi)
during_flood = collection.filterDate('2019-03-10', '2019-03-20').min().clip(aoi)

diff = before_flood.subtract(during_flood)
water_threshold = diff.gt(3)


In [4]:
# Export Rainfall and Flood Mask as TIFFs using Direct Request
def download_ee_image(image, aoi, scale, filename):
    out_path = project_root / 'data' / 'raw' / filename
    print(f'Generating URL for {filename} (scale={scale}m)...')
    url = image.getDownloadURL({
        'region': aoi,
        'scale': scale,
        'format': 'GEO_TIFF',
        'crs': 'EPSG:4326'
    })
    print('Downloading...')
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(out_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f'SUCCESS! Saved to {filename}\n')
    else:
        print(f'ERROR: Failed to save {filename}\n')

# Run downloads
download_ee_image(total_rainfall, aoi, scale=1000, filename='chirps_rainfall_mar2019.tif')
download_ee_image(water_threshold, aoi, scale=90, filename='sentinel_flood_mask_2019.tif')
print('All Remote Sensing files downloaded successfully!')


Generating URL for chirps_rainfall_mar2019.tif (scale=1000m)...
Downloading...
SUCCESS! Saved to chirps_rainfall_mar2019.tif

Generating URL for sentinel_flood_mask_2019.tif (scale=90m)...
Downloading...
SUCCESS! Saved to sentinel_flood_mask_2019.tif

All Remote Sensing files downloaded successfully!
